# Bài 0 · Cú pháp Keras và PyTorch

    Bài này chuyển các ví dụ ở trang 18–35 của **01 Introduction to Python, TensorFlow and Keras**. Các ô minh họa dùng tensor tạo tại chỗ. Phần cuối chạy thử quy trình huấn luyện và đánh giá bằng dữ liệu tổng hợp.

In [1]:
import os
import sys
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary

project = next((path for path in (Path.cwd(), *Path.cwd().parents)
                if (path / 'src/pytorch_core.py').is_file()), None)
if project is None:
    raise FileNotFoundError('Hãy mở notebook từ thư mục KerasToPytorch hoặc notebooks/')
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from src.pytorch_core import evaluate, fit, get_device, seed_everything

seed_everything()
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = get_device()
print('Phiên bản PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('Thiết bị chạy:', device)
print('GPU CUDA:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

Phiên bản PyTorch: 2.15.0+cpu
CUDA build: None
Thiết bị chạy: cpu
GPU CUDA: []


## Input và Dense · trang 23

    Keras mô tả mỗi ảnh bằng Input(shape=(28, 28, 1)). PyTorch nhận batch ảnh qua forward, với thứ tự NCHW. Dense(128, activation='relu') tương ứng Linear kết hợp ReLU. Linear áp dụng trên chiều cuối của đầu vào.

In [2]:
images_nhwc = torch.rand(4, 28, 28, 1)
images = images_nhwc.permute(0, 3, 1, 2)
dense = nn.Sequential(nn.Linear(784, 128), nn.ReLU())
print('NHWC → NCHW:', tuple(images_nhwc.shape), '→', tuple(images.shape))
print('Dense:', tuple(dense(images.flatten(1)).shape))

NHWC → NCHW: (4, 28, 28, 1) → (4, 1, 28, 28)
Dense: (4, 128)


## Conv2D, MaxPooling2D và Dropout · trang 18, 25–26

    Conv2D dùng padding='same' ở trang 18. Với kernel 3×3 và stride 1, padding=1 giữ cạnh ảnh. Ví dụ trang 25 bỏ đối số padding, tương ứng padding=0 và cạnh ảnh từ 28 xuống 26. MaxPool 2×2 sau đó làm cạnh 26 xuống 13. Dropout(0.75) có xác suất đặt một phần tử về 0 là 0,75 khi train.

In [3]:
same_conv = nn.Conv2d(1, 32, kernel_size=3, padding=1)
valid_conv = nn.Conv2d(1, 128, kernel_size=3)
pooled = nn.MaxPool2d(2)(valid_conv(images).relu())
drop = nn.Dropout(0.75)
probe = torch.ones(4096)
print('Conv same:', tuple(same_conv(images).relu().shape))
print('Conv valid + pool:', tuple(pooled.shape))
print('Tỷ lệ zero lúc train:', drop.train()(probe).eq(0).float().mean().item())
print('Dropout giữ nguyên tensor lúc eval:', torch.equal(drop.eval()(probe), probe))

Conv same: (4, 32, 28, 28)
Conv valid + pool: (4, 128, 13, 13)
Tỷ lệ zero lúc train: 0.739990234375
Dropout giữ nguyên tensor lúc eval: True


## Flatten và LSTM · trang 27

    Flatten gộp các chiều sau batch cho MLP. Với chuỗi ảnh đã trích đặc trưng, flatten từ chiều 2 để giữ trục thời gian. LSTM(64) trong slide lấy đầu ra ở bước cuối. Trong PyTorch dùng batch_first=True rồi lấy output[:, -1, :].

In [4]:
print('Flatten bản đồ đặc trưng:', tuple(nn.Flatten()(pooled).shape))
maps_over_time = torch.rand(4, 7, 8, 3, 3)
sequence = maps_over_time.flatten(start_dim=2)
lstm = nn.LSTM(input_size=72, hidden_size=64, batch_first=True)
output, (hidden, state) = lstm(sequence)
print('Chuỗi:', tuple(sequence.shape), '→', tuple(output.shape))
print('Bước cuối:', tuple(output[:, -1, :].shape))
assert torch.equal(output[:, -1, :], hidden[-1])

Flatten bản đồ đặc trưng: (4, 21632)
Chuỗi: (4, 7, 72) → (4, 7, 64)
Bước cuối: (4, 64)


## Mô hình đầy đủ và summary · trang 29–32

    Mô hình một nhánh được dựng bằng nn.Sequential, theo thứ tự Conv 64 kênh → Pool → Dropout 0,5 → Flatten → Dense 128 → Dense 10. Sau pooling có 64 × 13 × 13 = 10.816 phần tử. Mô hình trả logits để dùng CrossEntropyLoss.

In [5]:
model = nn.Sequential(
    nn.Conv2d(1, 64, kernel_size=3), nn.ReLU(), nn.MaxPool2d(2),
    nn.Dropout(0.5), nn.Flatten(),
    nn.Linear(64 * 13 * 13, 128), nn.ReLU(), nn.Linear(128, 10),
).to(device)
information = summary(model, input_size=(1, 1, 28, 28), device=device,
                      col_names=('input_size', 'output_size', 'num_params'))
assert information.total_params == 1_386_506

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
Sequential                               [1, 1, 28, 28]            [1, 10]                   --
├─Conv2d: 1-1                            [1, 1, 28, 28]            [1, 64, 26, 26]           640
├─ReLU: 1-2                              [1, 64, 26, 26]           [1, 64, 26, 26]           --
├─MaxPool2d: 1-3                         [1, 64, 26, 26]           [1, 64, 13, 13]           --
├─Dropout: 1-4                           [1, 64, 13, 13]           [1, 64, 13, 13]           --
├─Flatten: 1-5                           [1, 64, 13, 13]           [1, 10816]                --
├─Linear: 1-6                            [1, 10816]                [1, 128]                  1,384,576
├─ReLU: 1-7                              [1, 128]                  [1, 128]                  --
├─Linear: 1-8                            [1, 128]                  [1, 10]                   1,290
Total params: 1,386,506


## Compile, fit và evaluate · trang 33–35

    Compile được thay bằng hàm mất mát và bộ tối ưu. DataLoader quy định batch size. Ví dụ chạy hai epoch để minh họa luồng gọi API. Nhãn lấy từ tín hiệu tổng hợp gồm vị trí một điểm sáng trên ảnh. Đây là bài chạy minh họa API, còn kết quả MNIST nằm trong notebook 01 và 02.

In [6]:
example_labels = torch.arange(120) % 10
features = 0.05 * torch.rand(120, 1, 28, 28)
features[torch.arange(120), 0, 14, example_labels + 4] += 1
train = DataLoader(TensorDataset(features[:80], example_labels[:80]), batch_size=16, shuffle=True)
val = DataLoader(TensorDataset(features[80:100], example_labels[80:100]), batch_size=16)
test = DataLoader(TensorDataset(features[100:], example_labels[100:]), batch_size=16)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
history = fit(model, train, val, criterion, optimizer, device, epochs=2)
scores = evaluate(model, test, criterion, device)
print('Test minh họa:', scores)

Epoch 01/2 | train 2.3075, accuracy 0.1000 | val 2.3000, accuracy 0.1000 | lr 0.01
Epoch 02/2 | train 2.3055, accuracy 0.1125 | val 2.2963, accuracy 0.1000 | lr 0.01
Test minh họa: {'loss': 2.2962236404418945, 'accuracy': 0.15}
